
# Harpy-first SpatialData mIF test notebook (lazy TIFF read, no Sopa)

This version removes `sopa` and builds the `SpatialData` image directly from a
**lazy dask-backed TIFF reader** using:

- `tifffile`
- `dask.array`
- `dask.delayed`
- `spatialdata`
- `Image2DModel.parse`
- your channel map JSON

Goal:

1. Read a merged multichannel OME-TIFF without `sopa`
2. Build a **lazy** `SpatialData` object
3. Run **InstanSeg** through **Harpy**
4. Vectorize the resulting labels
5. Aggregate image intensities on the **label** layers
6. Write a SpatialData zarr store

This is closer to a real WSI path than the eager `tifffile.imread(...)` version because
the image is constructed as a dask array and channels are loaded lazily.


Configured for `SLIDE-0329_crop_2048` paths from your `spatialdata_crop_core_workflow_image2dmodel_SLIDE-0329.ipynb` notebook.

In [2]:

from pathlib import Path
import json
import os
import shutil
import sys

import numpy as np
import pandas as pd
import tifffile

from dask.distributed import Client, LocalCluster

import spatialdata as sd
from spatialdata import SpatialData, read_zarr
from spatialdata.models import Image2DModel

import harpy as hp

try:
    import torch
except Exception:
    torch = None

try:
    import cupy  # noqa: F401
    HAS_CUPY = True
except Exception:
    HAS_CUPY = False

try:
    import rasterio  # noqa: F401
    HAS_RASTERIO = True
except Exception:
    HAS_RASTERIO = False

print("Python:", sys.version.split()[0])
print("spatialdata:", getattr(sd, "__version__", "unknown"))
print("harpy:", getattr(hp, "__version__", "unknown"))
print("torch:", getattr(torch, "__version__", "not installed"))
print("cupy available:", HAS_CUPY)
print("rasterio available:", HAS_RASTERIO)


Python: 3.11.15
spatialdata: 0.4.0
harpy: 0.3.6
torch: 2.11.0+cu130
cupy available: False
rasterio available: True


In [8]:
# User parameters
CHANNEL_MAP_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/channel_map.generated.json")
SOURCE_SDATA_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_tiffslide_zarr_merged_SLIDE-0329.sdata.zarr/")

# InstanSeg torchscript model path used by Harpy workers
INSTANSEG_MODEL_PATH = Path("/mnt/c/Analysis/mIF-pipeline/Reference/instanseg-main/instanseg/utils/biological_images_datasets/instanseg.pt")

# Kept here for convenience / later extension, though this notebook does not use them directly yet
CELL_MASK_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/masks/SLIDE-0329_crop_2048_whole_cell.tiff")
NUCLEAR_MASK_PATH = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/masks/SLIDE-0329_crop_2048_nuclear.tiff")
NIMBUS_TABLE_PATH = Path("/mnt/c/Analysis/Data_prototype/nimbus_multislide/per_slide/SLIDE-0329_crop_2048/cell_table_full.csv")

OUTPUTS_DIR = Path("/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs")
WORKDIR = OUTPUTS_DIR / "harpy_lazy_test"
WORKDIR.mkdir(parents=True, exist_ok=True)

BACKED_ZARR_PATH = WORKDIR / "test_input_backed.zarr"
FINAL_ZARR_PATH = WORKDIR / "test_pipeline_output.zarr"

IMAGE_LAYER = "full_image"
SEG_IMAGE_LAYER = "segmentation_image"

CELL_LABELS = "instanseg_cells"
NUC_LABELS = "instanseg_nuclei"
CELL_SHAPES = "instanseg_cells_shapes"
NUC_SHAPES = "instanseg_nuclei_shapes"

CELL_TABLE = "table_cells"
NUC_TABLE = "table_nuclei"

PIXEL_SIZE_UM = 0.325
DEVICE = "cuda" if (torch is not None and torch.cuda.is_available()) else "cpu"

CHUNK_SHAPE = (1, 512, 512)

# Harpy tutorial-style segmentation settings
SEGMENT_CHUNKS = 1000

# Optional subset of channels used only for InstanSeg.
# Set to None to use all channels in IMAGE_LAYER.
SEGMENT_CHANNELS = [
    "R1_DAPI",
    "R2_DAPI",
    "R3_DAPI",
    "R4_P19_POLYRAT",
    "R4_GFP_POLY_AF488",
]

ALLOCATE_MODE = "sum"
ALLOCATE_OBS_STATS = ["var", "skew", "count"]
ALLOCATE_RUN_ON_GPU = HAS_CUPY
CELL_INSTANCE_SIZE_KEY = "cell_size"
NUC_INSTANCE_SIZE_KEY = "nucleus_size"

# Optional crop: [x_min, x_max, y_min, y_max]
DEBUG_CRD = None

required_paths = [SOURCE_SDATA_PATH, CHANNEL_MAP_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    print("Missing required inputs:", missing_paths)

print("DEVICE:", DEVICE)
print("ALLOCATE_RUN_ON_GPU:", ALLOCATE_RUN_ON_GPU)
print("CHUNK_SHAPE:", CHUNK_SHAPE)
print("OUTPUTS_DIR:", OUTPUTS_DIR)
print("WORKDIR:", WORKDIR)
print("INSTANSEG_MODEL_PATH:", INSTANSEG_MODEL_PATH)
print("ALLOCATE_MODE:", ALLOCATE_MODE)
print("ALLOCATE_OBS_STATS:", ALLOCATE_OBS_STATS)
print("SEGMENT_CHANNELS:", SEGMENT_CHANNELS)


DEVICE: cuda
ALLOCATE_RUN_ON_GPU: False
CHUNK_SHAPE: (1, 512, 512)
OUTPUTS_DIR: /mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs
WORKDIR: /mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/harpy_lazy_test
INSTANSEG_MODEL_PATH: /mnt/c/Analysis/mIF-pipeline/Reference/instanseg-main/instanseg/utils/biological_images_datasets/instanseg.pt
ALLOCATE_MODE: sum
ALLOCATE_OBS_STATS: ['var', 'skew', 'count']
SEGMENT_CHANNELS: ['R1_DAPI', 'R2_DAPI', 'R3_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488']


## Dask client

This follows the Harpy tutorial pattern more closely:

- use **1 worker on GPU**
- use more workers on CPU
- set an explicit memory limit

For GPU runs, it is usually better to increase chunk size only as far as fits on the GPU, while keeping `n_workers=1`.


In [5]:
print(f"Using device: {DEVICE}.")

cluster = LocalCluster(
    n_workers=4 if DEVICE == "cpu" else 1,
    threads_per_worker=1,
    memory_limit="20GB",
)

client = Client(cluster)

print(client.dashboard_link)
client


Using device: cuda.
http://127.0.0.1:8787/status


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 1
Total threads: 1,Total memory: 18.63 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:39153,Workers: 1
Dashboard: http://127.0.0.1:8787/status,Total threads: 1
Started: Just now,Total memory: 18.63 GiB
Comm: tcp://127.0.0.1:45801,Total threads: 1
Dashboard: http://127.0.0.1:38079/status,Memory: 18.63 GiB
Nanny: tcp://127.0.0.1:42239,



## Read OME-TIFF metadata and build a lazy dask image

This follows the lazy pattern you provided:

- read aliases from the channel map JSON
- inspect the OME series / pyramid metadata
- build a per-channel delayed reader
- stack channels into a lazy dask array
- derive multiscale factors from pyramid level shapes
- build a `SpatialData` image with `Image2DModel.parse`

This assumes channel-major storage where `series_shape[0]` is the channel count.


In [9]:
SOURCE_SDATA_PATH

PosixPath('/mnt/c/Analysis/Data_prototype/SLIDE-0329/outputs/spatialdata_image_write_only_tiffslide_zarr_merged_SLIDE-0329.sdata.zarr')

In [10]:

channel_map = json.loads(CHANNEL_MAP_PATH.read_text())
channel_aliases = [entry["alias"] for entry in channel_map]

sdata = read_zarr(SOURCE_SDATA_PATH)
img = hp.im.get_dataarray(sdata, IMAGE_LAYER)

print({
    "source_path": str(SOURCE_SDATA_PATH),
    "images": list(sdata.images.keys()),
    "image_layer": IMAGE_LAYER,
    "img_dims": tuple(img.dims),
    "img_shape": tuple(int(v) for v in img.shape),
    "img_chunks": img.chunks,
    "channel_count": len(img.coords["c"].values),
    "first_channels": list(map(str, img.coords["c"].values[:10])),
})

assert len(img.coords["c"].values) == len(channel_aliases), (
    "Channel map length must match the loaded SpatialData image channel count"
)


PathNotFoundError: nothing found at path ''

In [5]:
img


lazy_image_array: dask.array<rechunk-merge, shape=(25, 62617, 66406), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>
lazy_image_array.chunksize: (1, 512, 512)
scale_factors: [{'y': 2, 'x': 2}, {'y': 2, 'x': 2}, {'y': 2, 'x': 2}, {'y': 2, 'x': 2}, {'y': 2, 'x': 2}]


In [6]:
sdata


SpatialData object
└── Images
      └── 'full_image': DataTree[cyx] (25, 62617, 66406), (25, 31308, 33203), (25, 15654, 16601), (25, 7827, 8300), (25, 3913, 4150), (25, 1956, 2075), (25, 978, 1037), (25, 489, 518)
with coordinate systems:
    ▸ 'global', with elements:
        full_image (Images)

In [7]:

img = hp.im.get_dataarray(sdata, IMAGE_LAYER)
print(img)
print("dims:", img.dims)
print("shape:", img.shape)
print("n_channels:", len(img.coords["c"].values))
print("first channels:", list(map(str, img.coords["c"].values[:20])))


<xarray.DataArray 'image' (c: 25, y: 62617, x: 66406)> Size: 208GB
dask.array<rechunk-merge, shape=(25, 62617, 66406), dtype=uint16, chunksize=(1, 512, 512), chunktype=numpy.ndarray>
Coordinates:
  * c        (c) <U20 2kB 'R1_DAPI_AF' ... 'R4_GFP_POLY_AF488'
  * y        (y) float64 501kB 0.5 1.5 2.5 3.5 ... 6.261e+04 6.262e+04 6.262e+04
  * x        (x) float64 531kB 0.5 1.5 2.5 3.5 ... 6.64e+04 6.64e+04 6.641e+04
Attributes:
    transform:  {'global': Identity }
dims: ('c', 'y', 'x')
shape: (25, 62617, 66406)
n_channels: 25
first channels: ['R1_DAPI_AF', 'R1_BIT2_RS0584_CY3B', 'R1_BIT3_RS0015_CY5', 'R1_BIT4_RS0083_750', 'R1_DAPI', 'R1_BIT1_RS0996_488', 'R2_DAPI_AF', 'R2_BIT6_RS0639_CY3B', 'R2_BIT7_RS0109_CY5', 'R2_BIT8_RS0255_750', 'R2_DAPI', 'R2_BIT5_RS1047_488', 'R3_DAPI_AF', 'R3_FITC_AF', 'R3_BIT10_RS0763_CY3B', 'R3_BIT11_RS1312_CY5', 'R3_BIT12_RS0237_750', 'R3_DAPI', 'R3_BIT9_RS0805_488', 'R4_DAPI_AF']



## Optional quick read sanity check

This computes a very small spatial crop from one channel so you can confirm the lazy array is working
before writing / segmenting the full object.


In [8]:

# Optional tiny compute sanity check
SANITY_CHECK = True

if SANITY_CHECK:
    test_patch = img.data[0, :256, :256].compute()
    print("test_patch shape:", test_patch.shape, "dtype:", test_patch.dtype)


test_patch shape: (256, 256) dtype: uint16



## Back the SpatialData object to zarr


In [9]:
print("Using source SpatialData store directly:", SOURCE_SDATA_PATH)
print("Skipping intermediate image re-write to:", BACKED_ZARR_PATH)


In [10]:
print("Backed path:", sdata.path)
print("is_backed:", sdata.is_backed())


KeyboardInterrupt: 

2026-04-16 09:04:35,667 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2026-04-16 09:04:35,667 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/home/ratnayn/miniconda3/envs/harpy_instanseg/lib/python3.11/asyncio/runners.py", line 118, in run
    return self._loop.run_until_complete(task)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ratnayn/miniconda3/envs/harpy_instanseg/lib/python3.11/asyncio/base_events.py", line 654, in run_until_complete
    return future.result()
           ^^^^^^^^^^^^^^^
  File "/home/ratnayn/miniconda3/envs/harpy_instanseg/lib/python3.11/site-packages/distributed/nanny.py", line 985, in run
    await worker.finished()
  File "/home/ratnayn/miniconda3/envs/harpy_instanseg/lib/python3.11/site-packages/distributed/core.py", line 494, in finished
    await sel

## InstanSeg model path

Following the Harpy tutorial style, pass a **model path** to `hp.im.segment(...)` rather than a preloaded
`InstanSeg(...)` object. That allows each worker to load the model locally, which avoids serialization issues.


In [37]:
from instanseg import InstanSeg

if INSTANSEG_MODEL_PATH.exists():
    path_model = str(INSTANSEG_MODEL_PATH)
else:
    _ = InstanSeg("fluorescence_nuclei_and_cells", verbosity=1, device="cpu")
    path_model = os.path.join(
        os.environ.get("INSTANSEG_BIOIMAGEIO_PATH"),
        "fluorescence_nuclei_and_cells/0.1.1/instanseg.pt",
    )

print("path_model:", path_model)
print("path_model exists:", Path(path_model).exists())


Model fluorescence_nuclei_and_cells version 0.1.1 already downloaded in /home/ratnayn/miniconda3/envs/harpy_instanseg/lib/python3.11/site-packages/instanseg/utils/../bioimageio_models/, loading
path_model: /home/ratnayn/miniconda3/envs/harpy_instanseg/lib/python3.11/site-packages/instanseg/utils/../bioimageio_models/fluorescence_nuclei_and_cells/0.1.1/instanseg.pt
path_model exists: True


## Segment with Harpy + InstanSeg

This now follows the Harpy tutorial structure more closely:

- `hp.im.segment(...)` writes the aligned **labels** and **shapes** layers directly
- the segmentation input is created as a temporary view of the backed full image
- `labels_layer_align=CELL_LABELS`
- `instanseg_model=path_model` so the model is loaded inside each worker
- `chunks=SEGMENT_CHUNKS` set to the tutorial-style default of `1000`
- `output_shapes_layer=[NUC_SHAPES, CELL_SHAPES]`, matching the Harpy tutorial notebook

If `SEGMENT_CHANNELS` is set, the temporary segmentation object uses only those channels, but the final saved store keeps only the full image plus segmentation outputs.


In [38]:
full_image_da = hp.im.get_dataarray(sdata, IMAGE_LAYER)

if SEGMENT_CHANNELS is None:
    seg_image_da = full_image_da
else:
    available_channels = list(map(str, full_image_da.coords["c"].values))
    missing_segment_channels = [channel for channel in SEGMENT_CHANNELS if channel not in available_channels]
    assert not missing_segment_channels, f"Missing SEGMENT_CHANNELS in backed image: {missing_segment_channels}"
    seg_image_da = full_image_da.sel(c=SEGMENT_CHANNELS)

seg_image = Image2DModel.parse(
    seg_image_da.data,
    dims=tuple(seg_image_da.dims),
    c_coords=list(map(str, seg_image_da.coords["c"].values)),
)

seg_sdata = SpatialData(images={SEG_IMAGE_LAYER: seg_image})

seg_sdata = hp.im.segment(
    seg_sdata,
    img_layer=SEG_IMAGE_LAYER,
    output_labels_layer=[NUC_LABELS, CELL_LABELS],
    output_shapes_layer=[NUC_SHAPES, CELL_SHAPES],
    labels_layer_align=CELL_LABELS,
    chunks=SEGMENT_CHUNKS,
    model=hp.im.instanseg_callable,

    # parameters passed to hp.im.instanseg_callable
    output="all_outputs",
    device=DEVICE,
    instanseg_model=path_model,  # load in every worker; torchscript model is not serializable
    pixel_size=PIXEL_SIZE_UM,
    to_coordinate_system="global",
    scale_factors=None,
    overwrite=True,
)

sdata.labels[NUC_LABELS] = seg_sdata.labels[NUC_LABELS]
sdata.labels[CELL_LABELS] = seg_sdata.labels[CELL_LABELS]
sdata.shapes[NUC_SHAPES] = seg_sdata.shapes[NUC_SHAPES]
sdata.shapes[CELL_SHAPES] = seg_sdata.shapes[CELL_SHAPES]

del seg_sdata

print("Segmentation channels:", list(map(str, seg_image_da.coords["c"].values)))
print("Label layers:", list(sdata.labels.keys()))
print("Shape layers:", list(sdata.shapes.keys()))


2026-04-16 01:52:04.791 | INFO     | harpy.image.segmentation._segmentation:_segment:681 - Linking labels across chunks.
2026-04-16 01:52:04.836 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'instanseg_nuclei'
2026-04-16 01:52:04.840 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'instanseg_cells'
2026-04-16 01:52:04.841 | INFO     | harpy.image.segmentation._segmentation:_add_to_sdata:486 - Aligning labels layers: ['instanseg_nuclei', 'instanseg_cells']
2026-04-16 01:52:04.854 | INFO     | harpy.image.segmentation._map:_combine_dask_arrays:339 - Linking labels across chunks.
2026-04-16 01:52:04.880 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'instanseg_nuclei'
/home/ratnayn/miniconda3/envs/harpy_instanseg/lib/python3.11/site-packages/spatialdata/_core/_elements.py:88: UserWarning: Key `instanseg_nuclei` already exists. Overwriting it in-memory.
  self._check_key(key, self.keys(), self._shared_keys)
/h

Segmentation channels: ['R1_DAPI', 'R2_DAPI', 'R3_DAPI', 'R4_P19_POLYRAT', 'R4_GFP_POLY_AF488']
Label layers: ['instanseg_nuclei', 'instanseg_cells']
Shape layers: ['instanseg_nuclei_shapes', 'instanseg_cells_shapes']


In [39]:
print("Using chunks for allocation:", SEGMENT_CHUNKS)


Using chunks for allocation: 1000


## Aggregate intensities on labels

Keep this close to the Harpy API: pass `chunks=SEGMENT_CHUNKS` and let `allocate_intensity(...)` rechunk internally for aggregation.

This also follows the Harpy raster-aggregation notebook pattern:

- use `mode` for the main matrix stored in `.X`
- use `obs_stats` for extra shallow features stored in `.obs`
- keep `count` as an explicit instance-size column
- compute centers of mass so the output tables are immediately spatially usable


In [ ]:

sdata = hp.tb.allocate_intensity(
    sdata,
    img_layer=IMAGE_LAYER,
    labels_layer=CELL_LABELS,
    output_layer=CELL_TABLE,
    mode=ALLOCATE_MODE,
    obs_stats=ALLOCATE_OBS_STATS,
    instance_size_key=CELL_INSTANCE_SIZE_KEY,
    chunks=SEGMENT_CHUNKS,
    append=False,
    calculate_center_of_mass=True,
    run_on_gpu=ALLOCATE_RUN_ON_GPU,
    overwrite=True,
)

sdata = hp.tb.allocate_intensity(
    sdata,
    img_layer=IMAGE_LAYER,
    labels_layer=NUC_LABELS,
    output_layer=NUC_TABLE,
    mode=ALLOCATE_MODE,
    obs_stats=ALLOCATE_OBS_STATS,
    instance_size_key=NUC_INSTANCE_SIZE_KEY,
    chunks=SEGMENT_CHUNKS,
    append=False,
    calculate_center_of_mass=True,
    run_on_gpu=ALLOCATE_RUN_ON_GPU,
    overwrite=True,
)

print("Table layers:", list(sdata.tables.keys()))
print("Cell table shape:", sdata.tables[CELL_TABLE].shape)
print("Nuclear table shape:", sdata.tables[NUC_TABLE].shape)
display(sdata.tables[CELL_TABLE].to_df().head())
display(sdata.tables[CELL_TABLE].obs.head())
display(sdata.tables[NUC_TABLE].to_df().head())
display(sdata.tables[NUC_TABLE].obs.head())


## Optional visualization

Open the current `SpatialData` object in Napari for interactive inspection.


In [ ]:
from napari_spatialdata import Interactive

Interactive(sdata)



## Optional: add morphology features


In [ ]:

ADD_REGIONPROPS = False

if ADD_REGIONPROPS:
    sdata = hp.tb.add_regionprops(
        sdata,
        labels_layer=CELL_LABELS,
        table_layer=CELL_TABLE,
        output_layer=CELL_TABLE,
        overwrite=True,
    )

    sdata = hp.tb.add_regionprops(
        sdata,
        labels_layer=NUC_LABELS,
        table_layer=NUC_TABLE,
        output_layer=NUC_TABLE,
        overwrite=True,
    )



## Save the final SpatialData zarr


In [ ]:

if FINAL_ZARR_PATH.exists():
    import shutil
    shutil.rmtree(FINAL_ZARR_PATH)

sdata.write(FINAL_ZARR_PATH, overwrite=True)
print("Wrote:", FINAL_ZARR_PATH)


In [ ]:

sdata_final = sd.read_zarr(FINAL_ZARR_PATH)

print("images:", list(sdata_final.images.keys()))
print("labels:", list(sdata_final.labels.keys()))
print("shapes:", list(sdata_final.shapes.keys()))
print("tables:", list(sdata_final.tables.keys()))
sdata_final


## Close the Dask client

In [ ]:
client.close()


## Notes

What this improves over the eager version:

- the image is now dask-backed
- channels are loaded lazily
- pyramid-derived scale factors are preserved in the `Image2DModel`

Remaining caveats:

- this still reads **level 0 only** for the actual image data path used downstream
- the lazy reader assumes `key=channel_index` is the right way to address channels in your OME-TIFF
- for some OME layouts, axis handling may need to be adjusted if the file is not effectively `CYX` at level 0
